In [1]:
import pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier

df = pd.read_csv('../data/raw/diabetes_012_health_indicators_BRFSS2015.csv')
features = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'PhysActivity', 'GenHlth', 'Age', 'Sex']
X = df[features]
y = df['Diabetes_012'].replace({2: 1}).astype(int)   # 0 = no diabetes, 1 = prediabetes/diabetes

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [2]:
def evaluate(name, model):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(name, "\n", confusion_matrix(y_test, pred))
    return {
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'recall (at risk)': recall_score(y_test, pred),
        'precision (at risk)': precision_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, proba),
    }

results = []
rf = joblib.load('../models/chronic_model.pkl')
results.append(evaluate('Random Forest (current)', rf))

c:\Users\Khushi\Projects\ml-final-year-proj\AI-Healthcare-Risk-Prediction\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Khushi\Projects\ml-final-year-proj\AI-Healthcare-Risk-Prediction\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Random Forest (current) 
 [[31079 11662]
 [ 2770  5225]]


In [4]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg / pos, eval_metric='auc',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
results.append(evaluate('XGBoost', xgb))
pd.DataFrame(results).round(3)

XGBoost 
 [[29965 12776]
 [ 1764  6231]]


,model,accuracy,recall (at risk),precision (at risk),f1,roc_auc
0,Random Forest (current),0.716,0.654,0.309,0.420,0.750
1,XGBoost,0.713,0.779,0.328,0.462,0.817


In [5]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    'n_estimators': [200, 400, 600],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 5, 10],
}
search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=neg / pos, eval_metric='auc', random_state=42, n_jobs=-1),
    params, n_iter=20, scoring='roc_auc', cv=3, random_state=42, verbose=1
)
search.fit(X_train, y_train)
print("Best settings:", search.best_params_)

best_xgb = search.best_estimator_
results.append(evaluate('XGBoost (tuned)', best_xgb))
pd.DataFrame(results).round(3)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best settings: {'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
XGBoost (tuned) 
 [[29941 12800]
 [ 1743  6252]]


,model,accuracy,recall (at risk),precision (at risk),f1,roc_auc
0,Random Forest (current),0.716,0.654,0.309,0.420,0.750
1,XGBoost,0.713,0.779,0.328,0.462,0.817
2,XGBoost (tuned),0.713,0.782,0.328,0.462,0.818


In [6]:
joblib.dump(best_xgb, '../models/chronic_xgb_model.pkl')

['../models/chronic_xgb_model.pkl']

In [7]:
all_features = [c for c in df.columns if c != 'Diabetes_012']
X_all = df[all_features]
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

xgb_all = XGBClassifier(**search.best_params_, scale_pos_weight=neg / pos,
                        eval_metric='auc', random_state=42, n_jobs=-1)
xgb_all.fit(Xa_train, ya_train)

proba = xgb_all.predict_proba(Xa_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(confusion_matrix(ya_test, pred))
print("accuracy:", round(accuracy_score(ya_test, pred), 3),
      "recall:", round(recall_score(ya_test, pred), 3),
      "f1:", round(f1_score(ya_test, pred), 3),
      "roc_auc:", round(roc_auc_score(ya_test, proba), 3))

pd.Series(xgb_all.feature_importances_, index=all_features).sort_values(ascending=False)

[[30291 12450]
 [ 1711  6284]]
accuracy: 0.721 recall: 0.786 f1: 0.47 roc_auc: 0.825


HighBP                  0.543003
GenHlth                 0.122513
HighChol                0.081203
DiffWalk                0.046265
Age                     0.041127
BMI                     0.039925
HeartDiseaseorAttack    0.022069
CholCheck               0.018909
HvyAlcoholConsump       0.017101
Income                  0.016183
Sex                     0.012340
PhysHlth                0.005824
Education               0.005296
PhysActivity            0.005061
Stroke                  0.004786
MentHlth                0.003508
Smoker                  0.003506
Veggies                 0.003164
Fruits                  0.003074
NoDocbcCost             0.002647
AnyHealthcare           0.002493
dtype: float32

In [8]:
train_rows = set(map(tuple, X_train.values))
in_train = pd.Series([tuple(r) in train_rows for r in X_test.values], index=X_test.index)
print(f"{in_train.mean():.1%} of test rows also appear in training data")

p = best_xgb.predict_proba(X_test[~in_train])[:, 1]
print("ROC-AUC on rows never seen in training:", round(roc_auc_score(y_test[~in_train], p), 3))

93.3% of test rows also appear in training data
ROC-AUC on rows never seen in training: 0.78
